# Real Elliptic Curve

## 1. The Curve Equation and Singularity Check

An elliptic curve over the real numbers is defined by the Weierstrass equation:
$$y^2 = x^3 + ax + b$$

For the curve to form a valid, non-singular group (meaning it has no cusps or self-intersections), its discriminant must not be zero:
$$\Delta = -4a^3 - 27b^2 \neq 0$$

## 2. Deriving the Slope Formula ($m$)

To add two points $P(x_1, y_1)$ and $Q(x_2, y_2)$ geometrically, we draw a line connecting them. The slope $m$ of this line is calculated differently depending on whether the points are distinct.

### Case A: Point Addition ($P \neq Q$)
When adding two distinct points, we use standard algebra to find the slope of the secant line intersecting both points:
$$m = \frac{y_2 - y_1}{x_2 - x_1}$$

### Case B: Point Doubling ($P = Q$)
When adding a point to itself ($P + P$), we cannot use the secant formula since $x_1 = x_2$, which would result in division by zero. Instead, we compute the slope of the **tangent line** at point $P$ using implicit differentiation.

Taking the derivative of both sides of the curve equation $y^2 = x^3 + ax + b$ with respect to $x$:
$$2y \cdot \frac{dy}{dx} = 3x^2 + a$$

Solving for the derivative (the slope $m$):
$$\frac{dy}{dx} = \frac{3x^2 + a}{2y}$$

Evaluating this explicitly at our point $(x_1, y_1)$ provides the formula:
$$m = \frac{3x_1^2 + a}{2y_1}$$


## 3. Calculating the Resulting Point $R(x_3, y_3)$

Once we establish the slope $m$ (from either the secant or tangent line) passing through $P(x_1, y_1)$, the equation of the line is defined as:
$$y = m(x - x_1) + y_1$$

To find where this straight line intersects the elliptic curve a third time, we substitute the line equation into the curve equation:
$$(m(x - x_1) + y_1)^2 = x^3 + ax + b$$

Expanding the left side yields a polynomial where the highest-degree term is $m^2x^2$. Shifting all terms to one side generates a cubic equation:
$$x^3 - m^2x^2 + \dots = 0$$

According to **Vieta's Formulas**, the sum of the roots of a cubic equation $Ax^3 + Bx^2 + Cx + D = 0$ equals $-B/A$. Here, our three roots correspond to the x-coordinates of the curve intersections: $x_1, x_2,$ and $x_{int}$.
$$x_1 + x_2 + x_{int} = -(-m^2) = m^2$$

Isolating the new intersection x-coordinate ($x_3 = x_{int}$):
$$x_3 = m^2 - x_1 - x_2$$

To identify the corresponding y-coordinate of the intersection, we substitute $x_3$ back into the straight line equation:
$$y_{int} = m(x_3 - x_1) + y_1$$

Finally, geometric elliptic curve addition dictates that we **reflect** this third intersection point across the x-axis to obtain the final point $R$. This simply means negating the y-coordinate:
$$y_3 = -y_{int} = -(m(x_3 - x_1) + y_1) = m(x_1 - x_3) - y_1$$

In [2]:
import math

In [ ]:
class EllipticCurveReal:
    def __init__(self, a: float, b:float):
        # Check if the curve is singular
        if(abs(-4*a**3 - 27*b**2) <1e-9):
            raise ValueError("Curve is singular")
        
        self.a = a
        self.b = b

    def is_on_curve(self, P: tuple) -> bool:
        x = P[0]
        y = P[1]

        if(x == None and y == None):
            return True
        left_side = y**2
        right_side = x**3 + self.a * x + self.b

        return math.isclose(left_side, right_side, rel_tol=1e-9, abs_tol=1e-9)

    def add_points(self, p1: tuple, p2: tuple) -> tuple:
        """
        Args:
            p1 (tuple): The first point to add
            p2 (tuple): The second point to add

        Returns:
            tuple: The result of the addition
        """

        if not self.is_on_curve(p1):
            raise ValueError("Point not on curve")
        if not self.is_on_curve(p2):
            raise ValueError("Point not on curve")

        if p1 == (None, None):
            return p2
        if p2 == (None, None):
            return p1

        x1, y1 = p1
        x2, y2 = p2

        if math.isclose(x1, x2, rel_tol=1e-9, abs_tol=1e-9) and math.isclose(y1, -y2, rel_tol=1e-9, abs_tol=1e-9):
            return (None, None)

        if p1 != p2:
            m = (y2 - y1) / (x2 - x1)
        else:
            #Case when p1 == p2 and we use the 
            m = (3 * x1**2 + self.a) / (2 * y1)

        x3 = m**2 - x1 - x2 #By substituting m in the line equation
        y3 = m * (x3 - x1) + y1 #substitute x3 in the line equation to get y3 and obtain the (-) for reflection

        return (x3, -y3)


In [ ]:
class EllipticCurveFp:
    def __init__(self, a: int, b: int, p: int):
        # Check if the curve is singular
        if (4 * a**3 + 27 * b**2) % p == 0:
            raise ValueError("Curve is singular modulo p")
        
        self.p = p
        self.a = a
        self.b = b

    def binpow(self, a: int, b: int) -> int:
        """
        Args:
            a (int): The base
            b (int): The exponent

        Returns:
            int: The result of (a^b) mod p
        """
        if(b == 0):
            return 1
        elif(b%2 == 0):
            tmp = self.binpow(a, b//2, self.p) % self.p
            return tmp * tmp % self.p
        else:
            tmp = self.binpow(a, (b-1)//2, self.p) % self.p
            return tmp * a % self.p

    def modular_inverse(self, k: int) -> int:
        """
        Args:
            k (int): The number to find the modular inverse of
            p (int): The modulus

        Returns:
            int: The modular inverse of k modulo p
        """
        if k % self.p == 0:
            raise ValueError("No inverse exists")

        return self.binpow(k, self.p-2, self.p) % self.p

    def is_on_curve(self, P: tuple) -> bool:
        if P == (None, None):
            return True
        x = P[0]
        y = P[1]

        left = y*y % self.p
        right = ((x*x % self.p * x % self.p) + (self.a*x % self.p) + self.b) % self.p

        return left == right 

    def add_points(self, p1: tuple, p2: tuple) -> tuple:
        if not self.is_on_curve(p1):
            raise ValueError("Point not on curve")
        if not self.is_on_curve(p2):
            raise ValueError("Point not on curve")

        if p1 == (None, None):
            return p2
        if p2 == (None, None):
            return p1

        x1, y1 = p1
        x2, y2 = p2


        if((x1 == x2) and ((y1+y2)% self.p == 0)):
            return (None, None)

        if(p1 != p2):
            dy = (y2 - y1) % self.p
            dx = (x2 - x1) % self.p
            m = (dy * self.modular_inverse(dx, self.p)) % self.p
        else:
            dy = (3 * x1 * x1 + self.a) % self.p
            dx = (2 * y1) % self.p
            m = (dy * self.modular_inverse(dx, self.p)) % self.p

        x3 = (m*m - x1 - x2) % self.p
        y3 = ((m*(x1-x3) % self.p) -y1) % self.p

        return (x3, y3)

    
        

False